# 2. A masked autoencoder, one visible transformation at a time

This notebook trains a small 3D masked autoencoder (MAE) end to end on **segmented
synthetic cells**, not random noise. It is intentionally small enough for CPU execution,
but it uses the same package APIs, checkpoint format, metric events, and label-first
export as a larger experiment.

An MAE here is a **representation learner**, not a segmentation network: the cell mask
has already been produced upstream. The model hides 3D patches from a masked intensity
crop, predicts their cell-interior voxels, and uses its encoder output as an embedding.

## Step 0 — Central configuration and fixed seeds

The generated config is saved with the run. This cell also reports the actual compute
device; CUDA is optional and this tutorial deliberately requests CPU.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import torch
import yaml

from morphofeatures.artifacts import inspect_checkpoint, inspect_embedding
from morphofeatures.config import load_config, repository_root
from morphofeatures.data.crops import extract_masked_cell_crops, save_crop_batch
from morphofeatures.data.io import load_embeddings
from morphofeatures.data.synthetic import save_synthetic_dataset
from morphofeatures.mae3d import build_mae_model, encode_from_config, train_from_config
from morphofeatures.metrics import metric_series, read_metric_events
from morphofeatures.training_runtime import load_checkpoint

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = repository_root()
BASE_CONFIG = load_config(REPO_ROOT / "configs" / "default.yaml")
OUTPUT_DIR = BASE_CONFIG.paths.output_root / "notebooks" / "02_mae_smoke"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__)
print("CUDA visible:", torch.cuda.is_available())
print("Tutorial device: cpu")
print("Output:", OUTPUT_DIR)

## Step 1 — Start from raw intensity plus a cell segmentation

The preprocessing API centers a fixed window on each cell, rescales intensity according
to an explicit policy, and zeros voxels outside that cell. It also retains the unmasked
raw crop and binary mask for QC. Those companions are not hidden model state.

In [ ]:
fixture_dir = save_synthetic_dataset(OUTPUT_DIR / "synthetic", seed=SEED)
raw = np.load(fixture_dir / "raw.npy")
cells = np.load(fixture_dir / "cells.npy")

batch = extract_masked_cell_crops(
    raw,
    cells,
    crop_shape_zyx=(12, 12, 12),
    min_cell_voxels=100,
    min_crop_coverage=1.0,
    normalization="dtype",  # the synthetic float data are already in [0, 1]
    seed=SEED,
)
paths = save_crop_batch(
    batch,
    OUTPUT_DIR / "preprocessed",
    {"source": "deterministic synthetic fixture", "seed": SEED, "coordinate_order": "zyx"},
)
display(batch.manifest[["label_id", "voxel_count_roi", "crop_coverage", "mask_fraction"]])
print("model crop array:", batch.crops.shape, batch.crops.dtype)

## Step 2 — Inspect raw, mask, and model input together

The three columns below are different objects:

- **raw window** retains neighboring structures;
- **cell mask** says which voxels belong to this label;
- **model input** retains intensity only inside the target cell.

For real data, inspect several z slices and orthogonal planes. A centered crop can still
truncate a long process, and a good coverage number can still hide a segmentation merge.

In [ ]:
fig, axes = plt.subplots(len(batch.label_ids), 3, figsize=(9, 2.4 * len(batch.label_ids)), constrained_layout=True)
for row, label_id in enumerate(batch.label_ids):
    z = batch.crops.shape[1] // 2
    axes[row, 0].imshow(batch.raw_crops[row, z], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].imshow(batch.masks[row, z], cmap="gray")
    axes[row, 2].imshow(batch.crops[row, z], cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_ylabel(f"label {label_id}")
    for column, title in enumerate(["raw window", "cell mask", "masked input"]):
        if row == 0:
            axes[row, column].set_title(title)
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
plt.show()

## Step 3 — Why save a separate loss mask?

Most voxels in a cell-centered cube may be outside the cell. If ordinary MSE includes
those zeros, a model can lower loss by predicting background. MorphoFeatures therefore
accepts `data.loss_masks`: patches are still masked randomly, but reconstruction error
is accumulated only at foreground voxels inside masked patches. The mask is **not** an
extra encoder channel, so the input remains the historical masked-intensity modality.

In [ ]:
configuration = {
    "seed": SEED,
    "device": "cpu",
    "paths": {"repo_root": str(REPO_ROOT), "output_root": str(OUTPUT_DIR)},
    "data": {
        "crops": str(paths["crops"]),
        "label_ids": str(paths["label_ids"]),
        "loss_masks": str(paths["masks"]),
    },
    "mae": {
        "input_shape": [12, 12, 12],
        "patch_size": [4, 4, 4],
        "input_channels": 1,
        "embedding_dim": 16,
        "encoder_depth": 1,
        "encoder_heads": 4,
        "mask_ratio": 0.50,
    },
    "training": {
        "epochs": 3,
        "batch_size": 2,
        "learning_rate": 0.001,
        "validation_fraction": 0.25,
    },
    "inference": {"batch_size": 4},
}
config_path = OUTPUT_DIR / "mae_config.yaml"
config_path.write_text(yaml.safe_dump(configuration, sort_keys=False), encoding="utf-8")
display(pd.Series(configuration["mae"], name="value").to_frame())

## Step 4 — See the MAE patch grid and random mask before training

A `12³` crop with `4³` patches becomes a `3 × 3 × 3 = 27` token grid. At a mask ratio
of 0.5, 14 tokens are hidden. Patch size is physical: it must be chosen together with
voxel resolution, not copied blindly between datasets.

In [ ]:
preview_model = build_mae_model(configuration)
preview_input = torch.from_numpy(batch.crops[:1, None])
preview_loss_mask = torch.from_numpy(batch.masks[:1, None].astype(np.float32))
torch.manual_seed(SEED)
with torch.no_grad():
    preview_output = preview_model(preview_input, mask_ratio=0.50, loss_mask=preview_loss_mask)
preview_mask = preview_model.mask_volume(preview_output.mask, preview_input.shape)[0].numpy()

z = preview_input.shape[-3] // 2
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
axes[0].imshow(preview_input[0, 0, z], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("model input")
axes[1].imshow(preview_mask[z], cmap="Reds", vmin=0, vmax=1)
axes[1].set_title("masked MAE patches")
axes[2].imshow(preview_input[0, 0, z], cmap="gray", vmin=0, vmax=1)
axes[2].imshow(preview_mask[z], cmap="Reds", alpha=0.35, vmin=0, vmax=1)
axes[2].set_title("mask over input")
for axis in axes:
    axis.set_axis_off()
plt.show()

## Step 5 — Train through the package API

This cell intentionally resets only this tutorial's metrics file so rerunning it yields
one clean event series. Training writes the same portable checkpoint and append-only
JSONL events used by the CLI and Streamlit/SLURM monitor.

In [ ]:
checkpoint_path = OUTPUT_DIR / "checkpoint.pt"
metrics_path = OUTPUT_DIR / "metrics.jsonl"
if metrics_path.exists():
    metrics_path.unlink()

saved_checkpoint = train_from_config(config_path, output=checkpoint_path)
assert saved_checkpoint == checkpoint_path
print("checkpoint:", checkpoint_path)

In [ ]:
events = read_metric_events(metrics_path)
loss_frame = pd.DataFrame(metric_series(events))
display(loss_frame[["epoch", "train_loss", "validation_loss", "learning_rate"]])

axis = loss_frame.plot(
    x="epoch", y=["train_loss", "validation_loss"], marker="o", figsize=(7, 4)
)
axis.set_ylabel("foreground masked-patch MSE")
axis.set_title("Bookkeeping check on four synthetic cells")
axis.grid(alpha=0.3)
plt.show()

The curve verifies that optimization and validation bookkeeping work. Three epochs on
four synthetic cells do not establish convergence, generalization, or biological value.
Foreground-aware MSE is also not numerically comparable to the legacy contrastive loss.

## Step 6 — Inspect and reload the checkpoint

Portable checkpoints store unwrapped model weights plus epoch, step, config, and final
metrics. Reloading into a freshly constructed model is the test that matters; the Python
object left in memory after training is not an artifact.

In [ ]:
checkpoint_summary = inspect_checkpoint(checkpoint_path)
display(pd.Series({
    "epoch": checkpoint_summary["epoch"],
    "step": checkpoint_summary["step"],
    "parameter tensors": checkpoint_summary["parameter_tensors"],
    **checkpoint_summary["metrics"],
}).to_frame("value"))

reloaded_model = build_mae_model(configuration)
payload = load_checkpoint(checkpoint_path, reloaded_model, device="cpu")
reloaded_model.eval()
print("reloaded epoch:", payload["epoch"])

## Step 7 — View a masked prediction honestly

`reconstruction` contains predictions for every token, although the loss trains only
masked tokens. For visualization we keep visible input patches and insert predictions
only where the mask is true. Reconstruction quality is a diagnostic of the pretext task,
not the biological interpretation of the embedding.

In [ ]:
inputs = torch.from_numpy(batch.crops[:2, None])
loss_masks = torch.from_numpy(batch.masks[:2, None].astype(np.float32))
torch.manual_seed(SEED)
with torch.no_grad():
    output = reloaded_model(inputs, mask_ratio=0.50, loss_mask=loss_masks)
    composite = reloaded_model.composite_reconstruction(inputs, output)
    voxel_mask = reloaded_model.mask_volume(output.mask, inputs.shape)

sample, z = 0, inputs.shape[-3] // 2
hidden_input = inputs[sample, 0].clone()
hidden_input[voxel_mask[sample]] = 0
error = (composite[sample, 0] - inputs[sample, 0]).abs()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), constrained_layout=True)
for axis, image, title in zip(
    axes,
    [inputs[sample, 0, z], hidden_input[z], composite[sample, 0, z], error[z]],
    ["target", "visible patches", "composite prediction", "absolute error"],
):
    axis.imshow(image, cmap="magma" if title == "absolute error" else "gray", vmin=0)
    axis.set_title(title)
    axis.set_axis_off()
plt.show()

## Step 8 — Encode and validate the label-first result

Encoding uses the complete, unmasked crop. The output is sorted by `label_id`; no row
position from a DataLoader is treated as biological identity.

In [ ]:
embedding_path = OUTPUT_DIR / "embeddings.npy"
encode_from_config(config_path, checkpoint_path, embedding_path)
embedding = load_embeddings(embedding_path)
summary = inspect_embedding(embedding_path)
display(pd.Series(summary.__dict__).to_frame("value"))
print("IDs preserved:", np.array_equal(embedding.label_ids, batch.label_ids))
print("label-first matrix shape:", embedding.as_array().shape)
assert np.isfinite(embedding.features).all()

In [ ]:
coordinates = PCA(n_components=2).fit_transform(embedding.features)
fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(coordinates[:, 0], coordinates[:, 1], s=80)
for label_id, (x, y) in zip(embedding.label_ids, coordinates):
    axis.text(x, y, str(label_id), fontsize=9, ha="left", va="bottom")
axis.set_title("PCA of four synthetic MAE embeddings")
axis.set_xlabel("PC1")
axis.set_ylabel("PC2")
axis.grid(alpha=0.2)
plt.show()

## Legacy texture autoencoder versus proposed MAE

| Decision | Maintained legacy texture route | Proposed MAE route |
|---|---|---|
| model | 3D convolutional autoencoder | 3D patch tokenizer + Transformer |
| views | paired spatial/intensity augmentations | one crop with random token masks |
| objective | NT-Xent + full reconstruction + bottleneck | reconstruction on masked, cell-interior voxels |
| coarse input | cell/nucleus-centered masked volume | same biological unit, fixed patch-divisible window |
| fine input | many high-resolution patches, average by cell | train/encode patches, then aggregate by `label_id` |
| output | normally 80 dimensions | configurable; use 80 for six-group comparability |

The MAE does **not** remove the need for cell/nucleus mappings, physical-resolution
choices, coarse/fine sampling, or QC. It changes the self-supervised pretext task. A fair
comparison holds label cohort, resolution, crop policy, split, and downstream evaluation
constant.

## Scaling to the historical six groups

Train independent 80-dimensional models for cell shape, nucleus shape, coarse cell
texture, coarse nucleus texture, fine cell texture, and fine nucleus texture. Aggregate
fine patch embeddings per cell, join all six tables by `label_id`, require exact ID
alignment, then concatenate to 480 dimensions. Notebook 03 demonstrates the aligned
combination and downstream biological analysis; notebook 04 repeats this MAE workflow on
real EM and segmentation data while keeping its conclusions deliberately limited.